# 01 — Data Profiling

**Objective:** Load all raw tables and document the structure, quality, and data coverage before any transformation.

**Tables analyzed:**
- `orders` — 9,999 delivery orders
- `customers` — customer records
- `products` — product catalog
- `drivers` — delivery drivers
- `order_items` — items per order (junction table)

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.data_loader import load_all

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
tables = load_all()

for name, df in tables.items():
    print(f"{'='*50}")
    print(f"Table: {name.upper()} | Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
    print(f"{'='*50}")
    print(df.dtypes)
    print()

Table: ORDERS | Rows: 10,000 | Columns: 9
date               object
order_id           object
order_amount       object
region             object
items_delivered     int64
items_missing       int64
delivery_hour      object
driver_id          object
customer_id        object
dtype: object

Table: CUSTOMERS | Rows: 1,239 | Columns: 3
customer_id      object
customer_name    object
customer_age      int64
dtype: object

Table: PRODUCTS | Rows: 314 | Columns: 4
produc_id       object
product_name    object
category        object
price           object
dtype: object

Table: DRIVERS | Rows: 1,247 | Columns: 4
driver_id      object
driver_name    object
age             int64
Trips           int64
dtype: object

Table: ORDER_ITEMS | Rows: 1,662 | Columns: 2
order_id      object
product_id    object
dtype: object



## Null values per table

In [3]:
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print(f"{name}: no null values")
    else:
        print(f"\n{name}:")
        print(nulls)

orders: no null values
customers: no null values
products: no null values
drivers: no null values
order_items: no null values


## Duplicates

In [4]:
for name, df in tables.items():
    dups = df.duplicated().sum()
    print(f"{name}: {dups} duplicate row(s)")

orders: 0 duplicate row(s)
customers: 0 duplicate row(s)
products: 0 duplicate row(s)
drivers: 0 duplicate row(s)
order_items: 36 duplicate row(s)


## Sample of each table

In [5]:
for name, df in tables.items():
    print(f"\n--- {name.upper()} ---")
    display(df.head(3))


--- ORDERS ---


,date,order_id,order_amount,region,items_delivered,items_missing,delivery_hour,driver_id,customer_id
0,2023-01-01,c9da15aa-be24-4871-92a3-dfa7746fff69,"$1,095.54",Winter Park,10,1,8:37:28,WDID10627,WCID5031
1,2023-01-01,ccacc183-09f8-4fd5-af35-009d18656326,$659.11,Altamonte Springs,11,1,9:31:17,WDID10533,WCID5794
2,2023-01-01,f4e1d30b-c3d1-413f-99b8-93c0b46d68bf,$251.45,Winter Park,18,1,10:43:49,WDID10559,WCID5599



--- CUSTOMERS ---


,customer_id,customer_name,customer_age
0,WCID5170,Elijah Taylor,30
1,WCID5901,Alexis Ross,58
2,WCID5652,Carla Knox,23



--- PRODUCTS ---


,produc_id,product_name,category,price
0,PWPX0982761090982,Kellogg's Frosties,Supermarket,$12.53
1,PWPX0982761090983,Uncured Bacon,Supermarket,$4.67
2,PWPX0982761090984,Whole Milk,Supermarket,$9.95



--- DRIVERS ---


,driver_id,driver_name,age,Trips
0,WDID09873,Pamela Moore,18,64
1,WDID09874,Billy Lawson,18,37
2,WDID09875,Stephen Randolph,18,64



--- ORDER_ITEMS ---


,order_id,product_id
0,c7a343f7-3f1d-497c-8004-b9ede2d48fb1,PWPX0982761090982
1,c7a343f7-3f1d-497c-8004-b9ede2d48fb1,PWPX0982761090982
2,20698293-8399-4fda-af1e-b61a9ebb8a0a,PWPX0982761090983


## Issues identified

| Table | Column | Issue | Planned action |
|---|---|---|---|
| orders | `order_amount` | String format with `$` and `,` | Convert to float |
| orders | `delivery_hour` | Time string (`HH:MM:SS`) | Extract hour as integer |
| orders | `date` | String | Convert to datetime |
| products | `produc_id` | Typo in column name | Rename to `product_id` |
| products | `price` | String format with `$` | Convert to float |
| drivers | `Trips` | Uppercase name | Rename to `trips` |